In [ ]:
import pandas as pd
import numpy as np
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
export_dir = os.getcwd()
from pathlib import Path
import pickle
from collections import defaultdict
import time
import torch
import torch.nn as nn
import copy
import torch.nn.functional as F
import optuna
import logging
import matplotlib.pyplot as plt
import ipynb
import importlib
import sys
from pathlib import Path
from collections import Counter
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split 
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

from recbole.utils.case_study import full_sort_topk
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.trainer import Trainer
from recbole.utils import get_model, get_trainer


In [ ]:
# 1. Setup Configuration

parameter_dict = {
    'model': 'BPR',
    'dataset': 'my_dataset_LASTFM', # Ensure your .inter file is in ./dataset/lastfm/
    'data_path': '/home/mvarasteh/ProvidersExplantion/dataset/lastfm/',      # Ensure this points to the parent folder
    'epochs': 30,
    'stopping_step': 10,
    'train_batch_size': 256,
    'eval_args': {
        'split': {'RS': [0.7, 0.1, 0.2]}, # 80% Train, 10% Valid, 10% Test
        'group_by': 'user',
        'order': 'RO',
        'mode': 'full'},

    'load_col': {
#'inter': ['user_id', 'item_id', 'rating', 'timestamp']
'inter': ['user_id', 'item_id', 'playcount', 'timestamp']

   #'item': ['item_id', 'movie_title', 'release_year', 'genre'] # 'class' is the genre in ML-1M }
    
}}
config = Config(model='BPR', dataset='my_dataset_LASTFM', config_dict=parameter_dict)

# 2. Load Dataset
dataset = create_dataset(config)
print(f"Dataset loaded: {dataset}")

# 3. Split Dataset
# This returns three DataLoaders: train, validation, and test
train_data, valid_data, test_data = data_preparation(config, dataset)
print("Split complete: Train/Valid/Test loaders created.")

# 4. Initialize and Train Model
model = get_model(config['model'])(config, train_data.dataset).to(config['device'])
trainer = get_trainer(config['trainer'], config['model'])(config, model)

# Start training
trainer.fit(train_data, valid_data, saved=True, show_progress=True)

test_result = trainer.evaluate(test_data, load_best_model=False, show_progress=True)

print("Test Results:", test_result)

uid_list = torch.arange(1, dataset.user_num).to('cpu')

# 2. Get the Top 10 item IDs for these users
# This returns (scores, item_ids)
_, topk_iid_list = full_sort_topk(uid_list, model, test_data, k=10, device=config['device'])

# 3. Convert tensors to CPU/Numpy for processing
uid_list = uid_list.cpu().numpy()
topk_iid_list = topk_iid_list.cpu().numpy()

In [ ]:
topk_itms={}
for id, lists in zip((uid_list-1), (topk_iid_list-1)):
    topk_itms[id]=np.array(lists)

## Dataset and the Model

In [ ]:
data_name = "LASTFM" ### Can be ML1M, LASTFM
DP_DIR = Path("data/", data_name) 
export_dir = Path(os.getcwd())
files_path = Path(export_dir, DP_DIR)
checkpoints_path = Path(export_dir, "checkpoints")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
output_type_dict = {
    "VAE":"multiple",
    "MLP":"single"
}



hidden_dim_dict = {
    ("ML1M","VAE"): None,
    ("ML1M","MLP"): 128,

    ("Yahoo","VAE"): None,
    ("Yahoo","MLP"):32,
    
    ("Pinterest","VAE"): None,
    ("Pinterest","MLP"):512,
        ("LASTFM", "MLP"): 128,
                ("LASTFM", "VAE"): 128


}

## Training and Test Datasets

In [ ]:
## Load the data
train_data = pd.read_csv(Path(files_path,f'Train_Data_{data_name}_New.csv'), index_col=0)

test_data = pd.read_csv(Path(files_path,f'Test_Data_{data_name}_New.csv'), index_col=0)

#static_test_data = pd.read_csv(Path(files_path,f'Static_Test_Data_{data_name}_New.csv'), index_col=0)

## Load the items data and the popularity dictionary 
# items_data: meta features like genre

items_data = pd.read_csv(Path(files_path,f'Items_Data_{data_name}_New.csv'), index_col=0)
with open(Path(files_path,f'pop_dict_{data_name}.pkl'), 'rb') as f:
    pop_dict = pickle.load(f)

    
train_array = train_data.to_numpy()
test_array = test_data.to_numpy()
num_items=len(train_data.iloc[:,:-1].columns)
num_users=np.concatenate([train_array,test_array],axis=0).shape[0]

items_array = np.eye(num_items)
all_items_tensor = torch.Tensor(items_array).to(device)
all_array=np.concatenate((train_array, test_array))

## df for training and tes datasets without the users index
all_df_data=pd.concat([train_data,test_data], axis=0 )
#all_df_data.columns=all_df_data.columns.astype(int)

In [ ]:
items_data.rename(columns={'track_id':'item_id'}, inplace=True)

In [ ]:
pop_array = np.zeros(len(pop_dict))
for key, value in pop_dict.items():
    pop_array[key] = value

In [ ]:
from ipynb.fs.defs.recommenders_architecture import *
importlib.reload(ipynb.fs.defs.recommenders_architecture)
from ipynb.fs.defs.recommenders_architecture import *

In [ ]:
from ipynb.fs.defs.help_functions import *
importlib.reload(ipynb.fs.defs.help_functions)
from ipynb.fs.defs.help_functions import *


In [ ]:
## Converting each genre to a multihot vector 
ID2genre_hot=pd.concat([items_data["item_id"], items_data['genre'].str.get_dummies(sep='|')], axis=1)

In [ ]:
def genre(input_array, item2genre, users, Targ_genre):
    
    '''
    Counting the occurences of each genre in users profiles
    input_array: all users interactions
    item2genre: itmes to genre dataframe(multihot)
    users: all the users 
    Targ_genre: genres of the target item
    '''
    A=input_array[:,:-1]

    if not users:
        return 0
    
    else:
         
         z=sum( item2genre.set_index('item_id').loc[np.nonzero(A[uid])[0]].sum(axis=0) for uid in users)/len(users)
         
         return z[Targ_genre].mean()


In [ ]:
#items_data['year'] = items_data['name'].str.extract(r'\((\d{4})\)', expand=False).astype('int')
ID_Year=items_data.set_index('item_id')['year']
## Normalization
#max_val=ID_Year.max()
#min_val=ID_Year.min()
#ID_Year=(ID_Year-min_val)/(max_val-min_val)

In [ ]:
def recency_profiles(input_array, users, ID_Year):
    
    if not users:

        return 0  
    
    else: 
        avg_time_all=[]
        for u in users:
            user_profile=input_array[u,:-1]
            idx = np.nonzero(user_profile)[0]  # indices of interacted items

            avg_time=ID_Year[idx].mean()
            avg_time_all.append(avg_time)
        return np.mean(avg_time_all)


In [ ]:
users_size=(all_df_data. sum(axis=1).to_dict())

In [ ]:
sorted_pop=sorted(pop_dict.items(), key=lambda x:x[1], reverse=True)[:100]
Topk_pop, _=list(zip(*sorted_pop))

In [ ]:
recomms_list=np.concatenate(list(topk_itms.values()))
#mutul_itms=recomms_set.intersection(set(Topk_pop))


In [ ]:
recommendation_matrix = np.zeros((num_users, num_items), dtype=int)
for user, items in topk_itms.items():
    recommendation_matrix[user, items] = 1  

In [ ]:
## list of users for each item in recommendations
item2users=pd.DataFrame(list(topk_itms.items()), columns=['user_id', 'items_id']).explode('items_id').groupby('items_id')['user_id'].apply(list).to_dict()

In [ ]:
all_values=np.concatenate(list(topk_itms.values()))
items_indx=np.unique(all_values)
items_indx=np.arange( num_items)
count=Counter(all_values)
#count_all={item: count.get(item, 0) for item in items_indx}

In [ ]:
def profile_popularity (input_array, users):  
      
      '''
      computing average profiles populairty for users that recived the target item as a reocommendation 
      input_array: alll the interactions for each user and item
      users: all the users that recived the target item as a reocmmendation
      '''
      if not users:
        return 0
      
      else:
        All_pop = [] ## stores all pop scores for each users
    
        for u_id in users:
            
            ## profile for users
            user_profile=input_array[u_id,0:-1]
            ## get popularity of items in user profile
            mu=sum(pop_dict[int(item)] for item in np.nonzero(user_profile)[0]) / len(np.nonzero(user_profile)[0])
            All_pop.append(mu)

        return np.mean(All_pop)


In [ ]:

def profile_size(input_array, users, users_size):

    
    """
    input_array: np.array of shape [num_users, num_items+1]
                 Last column is user_id
    users: list of user_ids who received the target item as recommendation
    
    Returns:
        Average normalized profile size for the users
    """
   
    if not users:

        return 0
    
    else: 
    
        
        return sum(users_size[uid] for uid in users)/len(users)


In [ ]:

#cosine_sim_users=cosine_similarity(all_array[:,:-1])
cosine_sim_users=np.corrcoef(all_array[:,:-1])
mask=np.ones_like(cosine_sim_users)
np.fill_diagonal(mask,0)

similar_users_all = np.argsort(cosine_sim_users * mask)[:,::-1]

users_id=np.arange(similar_users_all.shape[0])
id_sim_users=similar_users_all[:,:10]
sim_users=dict(zip(users_id, id_sim_users))
df_sim_users=pd.DataFrame(sim_users.items(), index=None, columns=['user_id', 'similar_users'])

In [ ]:

def Targ_in_neighbs_profile (input_array, similar_users, users, targ_item, drop_last_col=True):
    """
    Count, for each user t, how many of t's similar users have targ_item (observed/consumed).
    input_array: 2D array [n_users, n_items( + maybe 1 label col)]
    similar_users: dict[int -> list[int]] of similar-user indices
    targ_item: int (column index in the **effective** item matrix)
    k: optional cap on number of similar users
    drop_last_col: set True if the last column is a label you want to ignore
    """
    if not users:
        return 0
    A = input_array[:, :-1] if drop_last_col else input_array
    # Boolean vector: does user u have targ_item?
    Target_has_item = A[:, targ_item] != 0

    dict_sim_users = dict(similar_users.iloc[users].values)
    
    counts = {}
    
    for t, S in dict_sim_users.items():
        #counts[t] = int(Target_has_item[np.asarray(S, dtype=int)].sum()) - int(Comp_has_item[np.asarray(S, dtype=int)].sum())
        counts[t] = int(Target_has_item[np.asarray(S, dtype=int)].sum()) 

    #for t, S in dict_sim_users.items():
     #   counts[t] = df_inter[(df_inter['track_id']==targ_item) & (df_inter['user_id'].isin(S))]['playcount'].sum()
    #min_val = min(counts.values())
    #max_val = max(counts.values())
    #range_val = (max_val - min_val)

    #Targ_in_neighb_prof = {k: (v - min_val) / range_val for k, v in counts.items()}
    
    #item2users_neighbprof=sum(counts[u] for u in item2users.get(targ_item,[0]) )/len (item2users.get(targ_item,[0]))
    item2users_neighbprof = sum(counts.values()) / len(counts)

    
    return item2users_neighbprof


In [ ]:

def Targ_in_neighbs_recomm(input_array, similar_users, users, targ_item, drop_last_col=False):

    """
    Count, for each item i, how many of i's similar users have targ_item (observed/consumed).
    input_array: 2D array [n_users, n_items( + maybe 1 label col)]
    similar_users: dict[int -> list[int]] of similar-user indices
    recommendations: dict[int -> list[int]] of recommended items for each user
    targ_item and Comp_item: int (column index in the **effective** item matrix)
    k: optional cap on number of similar users
    drop_last_col: set True if the last column is a label you want to ignore
    """
    if not users:
        return 0

    A = input_array[:, :-1] if drop_last_col else input_array
    # Boolean vector: does user u have targ_item?
    Target_has_item = A[:, targ_item] != 0


    dict_sim_users = dict(similar_users.iloc[users].values)
    counts = {}
    
    for t, S in dict_sim_users.items():
        #counts[t] = int(Target_has_item[np.asarray(S, dtype=int)].sum()) - int(Comp_has_item[np.asarray(S, dtype=int)].sum())
        counts[t] = int(Target_has_item[np.asarray(S, dtype=int)].sum()) 

    
    #min_val = min(counts.values())
    #max_val = max(counts.values())
    #range_val = (max_val - min_val)
    #print(f"range_val is {range_val}")
    #Targ_in_neighb_recomm = {k: (v - min_val) / range_val for k, v in counts.items()}


    #item2users_neighbrecomms=sum(counts[u] for u in item2users.get(targ_item,[0]) )/len (item2users.get(targ_item,[0]))
    
    item2users_neighbrecomms =  sum(counts.values()) / len(counts)


    return item2users_neighbrecomms




In [ ]:
#cosine_sim_items=cosine_similarity(all_array[:,:-1].T)
cosine_sim_items=np.corrcoef(all_array[:,:-1].T)
#items_embeddings=np.array(recommender.items_fc.weight.detach())
#cosine_sim=cosine_similarity(items_embeddings.T)
mask=np.ones_like(cosine_sim_items)
np.fill_diagonal(mask,0)
similar_items_all = np.argsort(cosine_sim_items * mask)[:,::-1]
itms_id=np.arange(similar_items_all.shape[0])
id_sim_itms=similar_items_all[:,:10]
sim_itms=dict(zip(itms_id, id_sim_itms))

In [ ]:

def neighbors_item_profile(input_array, similar_items, users, targ_item, drop_last_col=True):
    """
    For each user u, count how many of u's items are in targ_item's similar-items list.
    input_array: 2D array [n_users, n_items (+ maybe 1 user-id col)]
                 Assumes item columns are binary (0/1).
    similar_items: dict[int -> list[int]]  (item -> similar item indices)
    targ_item: int (column index in the effective item matrix)
    drop_last_col: if True, treat the last column of input_array as a user-id (ignored for items)
    """
    A = input_array[:, :-1] if drop_last_col else input_array
    user_ids = input_array[:, -1].astype(int) if drop_last_col else np.arange(A.shape[0])

    # indices of items similar to targ_item
    sim_idx = np.array(similar_items.get(targ_item, []), dtype=int)

    if len(similar_items[targ_item]) == 0:
        return 0

    # Ensure binary (handles 0/1 or counts)
    Ab = (A != 0)

    # Count, per user, how many of their items intersect sim_idx (vectorized slice + sum)
    counts_vec = Ab[:, sim_idx].sum(axis=1)
    count={int(u): int(c) for u, c in zip(user_ids, counts_vec)}

    # Return dict[user_id] -> count
    #min_val = min(count.values())
    #max_val = max(count.values())
    #range_val = (max_val - min_val)

    #Neighb_items_in_prof = {k: (v - min_val) / range_val for k, v in count.items()}
    
    item2users_neighbritms = 0 if not users else sum(count[u] for u in users) / len(users)

    
    return item2users_neighbritms


In [ ]:
ID_energy=items_data.set_index('item_id')['energy']

def energy_profiles(input_array, users, ID_energy):
    
    if not users:

        return 0
    
    else: 
        avg_all=[]
        for u in users:
            user_profile=input_array[u,:-1]
            idx = np.nonzero(user_profile)[0]  # indices of interacted items

            avg=ID_energy[idx].mean()
            avg_all.append(avg)
        return np.mean(avg_all)
    

    
ID_liveness=items_data.set_index('item_id')['liveness']

def liveness_profiles(input_array, users, ID_liveness):
    
    if not users:
        return 0  
    else: 
        avg_all=[]
        for u in users:
            user_profile=input_array[u,:-1]
            idx = np.nonzero(user_profile)[0]  # indices of interacted items

            avg=ID_liveness[idx].mean()
            avg_all.append(avg)
        return np.mean(avg_all)
    

ID_dance=items_data.set_index('item_id')['danceability']
ID_instrument=items_data.set_index('item_id')['instrumentalness']

def dance_profiles(input_array, users, ID_dance):
    
    if not users:

        return 0
    
    else: 
        avg_all=[]
        for u in users:
            user_profile=input_array[u,:-1]
            idx = np.nonzero(user_profile)[0]  # indices of interacted items

            avg=ID_dance[idx].mean()
            avg_all.append(avg)
        return np.mean(avg_all)
    
def instrument_profiles(input_array, users, ID_instrument):
    
    if not users:
        return 0

    else: 
        avg_all=[]
        for u in users:
            user_profile=input_array[u,:-1]
            idx = np.nonzero(user_profile)[0]  # indices of interacted items

            avg=ID_instrument[idx].mean()
            avg_all.append(avg)
        return np.mean(avg_all)


In [ ]:


df=pd.DataFrame()


num_rows=num_items
idx= torch.randperm(num_rows)[:1000]
#samples=topk_itms[idx]
count_pos=0
count_neg=0


over_expos_itms = np.array(list(item2users.keys()))  # <-- convert set to list first
under_expos_itms = list(set(items_data['item_id'].unique()) - set(over_expos_itms)) # -1 --. MovieIDs in Ratings_df starts from 1

# Sample safely
sample_size =  0
sampled_items = np.random.choice(under_expos_itms, size=sample_size, replace=False)

# Concatenate 1D arrays
all_items = np.concatenate([sampled_items, over_expos_itms])
rows_list = []  # Use a list for speed
for i in all_items:
        try:
                i=int(i)
                item_users=item2users.get(i,[])
                itm_genre=items_data.loc[items_data['item_id']==i,'genre' ].str.split('|').iloc[0]
                row = {
                        
                        'TargID': i,
                        #'playcounts': playcounts.get(i, 0),
                        #'profile_playcount': profile_playcount(all_array, item_users),
                        #'Artist Popularity': pop_artist[item2artist[i]],
                        #'Artis_prof_pop': Artist_profilr_pop(all_array, item_users),
                        #'energy_level_profile': energy_profiles(all_array, item_users, ID_energy),
                        #'liveness_profile': liveness_profiles(all_array, item_users, ID_liveness),
                        #'liveness':ID_liveness[i],
                        #'profiles_dance': dance_profiles(all_array, item_users, ID_dance),
                        #'target_dance': ID_dance[i],
                        #'instrument_profile': instrument_profiles(all_array, item_users, ID_instrument),
                        #'Profiles Time':recency_profiles(all_array,item_users, ID_Year),
                        #'Profiles Popularity': profile_popularity(all_array, item_users),

                         ## New features
                        'POP':pop_dict[i],
                        'SZ':profile_size(all_array, item_users, users_size),
                        'YR':ID_Year[i],
                        'GD':genre(all_array, ID2genre_hot, item_users, itm_genre ),

                        'ND': Targ_in_neighbs_profile (all_array, df_sim_users, item_users, i, drop_last_col=True),
                        'RI': neighbors_item_profile(all_array, sim_itms, item_users, i, drop_last_col=True),

                        'AE':ID_energy[i],
                        'INS':ID_instrument[i],

                        

                        'Exposure': count[i]
                                    }
                rows_list.append(row)
        except Exception as e:
                print(f"Skipping item {i} due to error: {e}")
        continue

#df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
df = pd.DataFrame(rows_list)
        #indx=(torch.randperm(49)+1)[0:5]

   
        


In [ ]:
df

In [ ]:
df['Exposure_']=np.log1p(df['Exposure'])


In [ ]:
scaler=MinMaxScaler()
array_scaled=scaler.fit_transform(df)

#df_scaled=pd.DataFrame(array_scaled, columns=df.columns)
df_scaled=pd.DataFrame(df, columns=df.columns)
#df_scaled['exposure_bin'] = pd.cut(df_scaled['Exposure'], bins=[0.0, 0.1, 0.3, 1.0], labels=['Low', 'Mid', 'High'],include_lowest=True)

X=df_scaled.drop(columns=['Exposure','TargID', 'Exposure_'])
y=df_scaled['Exposure_']


In [ ]:
import xgboost as xgb

results = []
X_filt = pd.DataFrame()

# 1. Define the objective once (outside the loop)
def asymmetric_objective(y_true, y_pred):
    residual = y_true - y_pred
    penalty_ratio = 5
    grad = np.where(residual > 0, -2 * penalty_ratio * residual, -2 * residual)
    hess = np.where(residual > 0, 2 * penalty_ratio, 2.0)
    return grad, hess

# 2. Iterate through features one by one
for f in X.columns.tolist():
    # Add the current feature to our subset
    X_filt[f] = X[f]
    
    # CRITICAL: Split only the filtered features (X_filt)
    X_train, X_test, y_train, y_test = train_test_split(
        X_filt, y, test_size=0.2, random_state=44, shuffle=True
    )

    # Initialize model
    regr = xgb.XGBRegressor(
        objective=asymmetric_objective, # Pass your custom objective here
        n_estimators=1000,
        learning_rate=0.01,
        max_depth=7,
        random_state=44
    )

    # 3. Fit on the subset of features
    regr.fit(X_train, y_train)

    # 4. Evaluate
    y_hat_log = regr.predict(X_test)
    log_r2 = r2_score(y_test, y_hat_log)
    mse = mean_squared_error(y_test, y_hat_log)
    
    # Store results for your plot
    results.append({'feature_added': f, 'R2': log_r2, 'MSE': mse})
    
    print(f"Added {f} -> Current R2: {log_r2:.4f}")

# 5. Convert to DataFrame for easy plotting
df_results = pd.DataFrame(results)

In [ ]:
df_results.to_csv(f"LASTFM_BPR.csv")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=44, shuffle=True
)


In [ ]:
import numpy as np
import xgboost as xgb
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

def asymmetric_objective(y_true, y_pred):
    # Calculate the residual
    residual = y_true - y_pred
    
    # Define your penalty ratio (e.g., 5.0 means under-predicting is 5x worse)
    penalty_ratio = 5
    
    # Gradient: If residual > 0 (under-predicted), multiply by penalty
    grad = np.where(residual > 0, -2 * penalty_ratio * residual, -2 * residual)
    
    # Hessian: Second derivative (constant for squared error)
    hess = np.where(residual > 0, 2 * penalty_ratio, 2.0)
    
    return grad, hess

# Initialize the model with the custom objective
regr = xgb.XGBRegressor(
    #objective=asymmetric_objective, # Pass your custom objective here

    n_estimators=1000,
    learning_rate=0.01,
    max_depth=7,
    random_state=44
)

# Apply the custom objective during fitting
# 2. Fit with Weights (XGBoost handles these very effectively)
regr.fit(X_train, y_train )

# 3. Predict and Evaluate
y_hat_log = regr.predict(X_test)
log_r2 = r2_score(y_test, y_hat_log)
print(f"XGBoost Log-space R2: {log_r2}")
MSE=mean_squared_error(y_test,y_hat_log )
print(f"XGBoost MSE:{MSE}")
# 4. Transform back for the Residual Plot
y_hat_original = np.expm1(y_hat_log) 
y_test_original = np.expm1(y_test)



residuals = y_test_original - y_hat_original

 #6. Plotting Residuals in Original Space
plt.figure(figsize=(10, 6))
plt.scatter(y_hat_original, residuals, alpha=0.5)
plt.axhline(0, color='r', linestyle='--')
plt.xlabel('Predicted Exposure (Original Scale)')
plt.ylabel('Residuals (Error)')
#plt.title(f'Model name: {recommender_name}')
plt.show()

In [ ]:
list(zip(regr.feature_names_in_, regr.feature_importances_))

In [ ]:


def asymmetric_objective(y_true, y_pred):
    # Calculate the residual
    residual = y_true - y_pred
    
    # Define your penalty ratio (e.g., 5.0 means under-predicting is 5x worse)
    penalty_ratio = 5
    
    # Gradient: If residual > 0 (under-predicted), multiply by penalty
    grad = np.where(residual > 0, -2 * penalty_ratio * residual, -2 * residual)
    
    # Hessian: Second derivative (constant for squared error)
    hess = np.where(residual > 0, 2 * penalty_ratio, 2.0)
    
    return grad, hess

# Initialize the model with the custom objective
regr = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=7,
    random_state=44
)


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=44, shuffle=True)

# Apply the custom objective during fitting
# 2. Fit with Weights (XGBoost handles these very effectively)
regr.fit(X_train, y_train )

# 3. Predct and Evaluate
y_hat_log = regr.predict(X_test)
log_r2 = r2_score(y_test, y_hat_log)
print(f"XGBoost Log-space R2 Full Model: {log_r2}")
backward_selection_df=pd.DataFrame([{'feature': 'full', 'R2':log_r2 }])

X_filt=X.copy()
feat_names=['POP', 'ND', 'RI', 'SZ', 'YR', 'GD', 'AE', 'INS']
for i in feat_names[::-1]:

    X_filt=X_filt.drop(columns=[i])
    X_train, X_test, y_train, y_test = train_test_split(
        X_filt, y, test_size=0.2, random_state=44, shuffle=True)

    # Apply the custom objective during fitting
    # 2. Fit with Weights (XGBoost handles these very effectively)
    regr.fit(X_train, y_train )

    # 3. Predict and Evaluate
    y_hat_log = regr.predict(X_test)
    log_r2 = r2_score(y_test, y_hat_log)
    backward_selection_df = pd.concat([
    backward_selection_df, 
    pd.DataFrame([{'feature': i, 'R2': log_r2}])
], ignore_index=True)


    print(f"XGBoost Log-space R2 Dropping {i}: {log_r2}")
    MSE=mean_squared_error(y_test,y_hat_log )
    print(f"XGBoost MSE:{MSE}")
    # 4. Transform back for the Residual Plot
    y_hat_original = np.expm1(y_hat_log) 
    y_test_original = np.expm1(y_test)



residuals = y_test_original - y_hat_original

#6. Plotting Residuals in Original Space
plt.figure(figsize=(10, 6))
plt.scatter(y_hat_original, residuals, alpha=0.5)
plt.axhline(0, color='r', linestyle='--')
plt.xlabel('Predicted Exposure (Original Scale)')
plt.ylabel('Residuals (Error)')
#plt.title(f'Model name: {recommender_name}')
plt.show()


In [ ]:
backward_selection_df.to_csv(f"backward_selection_results_{'BPR'}_{data_name}.csv", index=False)

In [ ]:
backward_selection_results_

In [ ]:
backward_selection_df